# Choosing `beta_max` for the faithful I2SB schedule

`kind="i2sb"` builds the paper's mirrored-quadratic betas, and **`beta_max` is its only scale knob**
(`tau` is ignored in that mode). This notebook works out what value to use for BraTS T1 -> T1ce,
empirically.

## The idea

At inference `x1` is **known**. The bridge state is

$$x_t = \mu_0(t)\,x_0 + \mu_1(t)\,x_1 + \sigma_{sb}(t)\,\varepsilon$$

so subtracting the known prior term and rescaling gives a sufficient statistic for $x_0$:

$$u(t) \;=\; \frac{x_t - \mu_1(t)\,x_1}{\mu_0(t)} \;=\; x_0 + \sigma_{\mathrm{eff}}(t)\,\varepsilon,
\qquad \boxed{\;\sigma_{\mathrm{eff}}(t) = \frac{\sigma_{sb}(t)}{\mu_0(t)}\;}$$

**Every step of the bridge is exactly a Gaussian denoising problem on $x_0$ at noise level
$\sigma_{\mathrm{eff}}(t)$, with $x_1$ available as side information.** So choosing `beta_max` =
choosing the *noise ladder* the regressor is trained on, and that ladder has to be matched to one
number: how far apart the two contrasts actually are,

$$\varsigma = \mathrm{RMS}(x_0 - x_1) \quad\text{(brain-masked, measured on real data)}.$$

* $\sigma_{\mathrm{eff}}(t) \gg \varsigma$ &rarr; $u(t)$ is *worse* than just answering $x_1$. Those
  steps teach nothing.
* $\sigma_{\mathrm{eff}}(t) \ll \varsigma$ &rarr; $u(t)$ hands over $x_0$. Those steps are trivial.
* The schedule earns its keep where $\sigma_{\mathrm{eff}}(t) \approx \varsigma$.

The criterion below picks `beta_max` so the crossover sits mid-schedule, then checks that choice
against real BraTS slices with two training-free estimators.

**Parts 1-2 need no data** (pure schedule algebra, runs anywhere). **Parts 3-6 read BraTS.**

In [ ]:
import os, json, math
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

# os.chdir('/scratch/ee2178/ImMAP')     # <-- EDIT to your repo root if not already there

from sb.base import build_schedule, bridge_coeffs, n_steps

CONFIG    = "config/BraTS/i2sb_gcdl_t1.json"   # the run this notebook is sizing
N_POINTS  = 1000                                # cfg["i2sb"]["n_points"]
BETA_GRID = [0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 4.0, 8.0]
BETA_REF  = 0.3                                 # the I2SB paper default, for reference lines

# BraTS (Parts 3-6). Read from the config so this tracks the run being sized.
try:
    _cfg     = json.load(open(CONFIG))
    VAL_ROOT = _cfg["data"]["val"]["root"]
    X0_IDX   = _cfg["data"]["val"]["x0_idx"]     # T1ce
    X1_IDX   = _cfg["data"]["val"]["x1_idx"]     # T1
    N_POINTS = _cfg["i2sb"]["n_points"]
    print(f"from {CONFIG}: x1=ch{X1_IDX} -> x0=ch{X0_IDX}, n_points={N_POINTS}\n  root={VAL_ROOT}")
except Exception as e:
    VAL_ROOT, X0_IDX, X1_IDX = None, 2, 1
    print(f"[warn] could not read {CONFIG} ({e}); Parts 3-6 need VAL_ROOT set by hand")

N_SUBJECTS = 12        # subjects to sample for the statistics
SLICE_STEP = 8         # take every Nth slice
MIN_BRAIN  = 0.02      # skip slices whose brain mask covers less than this fraction
MAX_VOX    = 400_000   # voxels subsampled per region for the Part 5/6 estimator sweeps
torch.manual_seed(0); np.random.seed(0)

## Part 1 - what `beta_max` actually changes

`i2sb_betas` is `linspace(sqrt(1e-4), sqrt(beta_max/n), n)**2`, first half mirrored. Note the fixed
`linear_start = 1e-4`: at `n=1000`, `beta_max = 0.1` makes the two endpoints **equal**, so the ramp
degenerates to a uniform schedule, and below 0.1 the betas ramp *downward*. That makes ~0.1 a floor
rather than a usable setting -- and it means the total injected variance has a floor too, which
Part 4 has to respect.

In [ ]:
scheds = {b: build_schedule(kind="i2sb", n_points=N_POINTS, beta_max=b) for b in BETA_GRID}
t = np.arange(1, N_POINTS + 1) / N_POINTS

print(f"{'beta_max':>9} {'total var':>10} {'max std_fwd':>12} {'peak std_sb':>12} {'= brownian tau':>15}")
for b, s in scheds.items():
    _, _, std_sb = bridge_coeffs(s)
    # peak std_sb IS the brownian tau by definition (brownian: max std_sb = tau at t=1/2),
    # so this column translates a beta_max into the tau knob used by the other runs.
    print(f"{b:>9.2f} {float(s.std_fwd[-1]**2):>10.4f} {float(s.std_fwd[-1]):>12.4f} "
          f"{float(std_sb.max()):>12.4f} {float(std_sb.max()):>15.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for b, s in scheds.items():
    _, _, std_sb = bridge_coeffs(s)
    ax[0].plot(t, s.std_fwd.numpy(), label=f"{b:g}")
    ax[1].plot(t, std_sb.numpy(), label=f"{b:g}")
ax[0].set_title(r"$\sigma_{fwd}(t)$  (what the net conditions on)")
ax[1].set_title(r"$\sigma_{sb}(t)$  (bridge noise injected)")
for a in ax:
    a.set_xlabel("t"); a.grid(alpha=.3); a.legend(title="beta_max", fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

## Part 2 - the effective denoising ladder

$\sigma_{\mathrm{eff}} = \sigma_{sb}/\mu_0$. This is the quantity `beta_max` really controls, and it
is what has to be compared against the data. It sweeps from `std_fwd[0]` (= 0.01, the schedule's
`linear_start`) up to $\infty$ as $t \to 1$, where the state is pure prior.

In [ ]:
def sigma_eff(sched):
    """sigma_sb / mu_x0 -- noise level of the equivalent x0-denoising problem at each step."""
    mu0, _, std_sb = bridge_coeffs(sched)
    return (std_sb / mu0).numpy()


def crossover_t(sched, s):
    """t where sigma_eff first exceeds s; past it the prior x1 beats the bridge state."""
    se = sigma_eff(sched)
    return (int(np.argmax(se > s)) + 1) / len(se) if np.any(se > s) else 1.0


fig, ax = plt.subplots(figsize=(6.5, 4))
for b, s in scheds.items():
    ax.semilogy(t, sigma_eff(s), label=f"{b:g}")
ax.set_xlabel("t"); ax.set_ylabel(r"$\sigma_{eff}$"); ax.grid(alpha=.3, which="both")
ax.set_title(r"effective denoising noise level $\sigma_{eff}(t)=\sigma_{sb}/\mu_0$")
ax.legend(title="beta_max", fontsize=8, ncol=2); plt.tight_layout(); plt.show()

# For the constant Brownian bridge var_fwd = V*t exactly, so sigma_eff = sqrt(V*t/(1-t)) and the
# crossover sigma_eff = s sits at t* = s^2/(s^2+V), V = total variance = std_fwd[-1]^2. In other
# words a schedule is "centred" (t*=1/2) on data whose residual RMS equals sqrt(V). The i2sb betas
# are not exactly linear in t, so Part 4 solves for t* numerically rather than using that formula.
V_ref = float(scheds[BETA_REF].std_fwd[-1] ** 2)
print(f"beta_max={BETA_REF}: total variance V={V_ref:.4f}, sqrt(V)={math.sqrt(V_ref):.4f}")
print(f"-> the paper default is centred on data with RMS residual ~= {math.sqrt(V_ref):.3f}")

## Part 3 - measure the data scale on real BraTS

$\varsigma = \mathrm{RMS}(x_0 - x_1)$ over the brain mask. Also measured **inside the enhancing
tumor**, because that is where T1ce actually differs from T1 -- and it is a much larger number, so
the whole-brain and ET-optimal schedules are not the same. (That same gap is what `et_weight` in
`train_i2sb` upweights; `i2sb_gcdl_t1_et.json` is the run that uses it.)

We read the h5 directly rather than through the loader: it keeps this notebook independent of
`et_mask`, and it lets us pull whole volumes instead of augmented crops.

In [ ]:
import h5py
from datasets.BraTS.i2sb_dataset import index_img_from_root


def sample_slices(root, n_subjects=N_SUBJECTS, step=SLICE_STEP, min_brain=MIN_BRAIN):
    """-> x0, x1, mask, et as (N,H,W) float32, in the stored z-scored space (scales are 1.0)."""
    paths, _ = index_img_from_root(root)
    X0, X1, MK, ET = [], [], [], []
    n_missing = 0
    for p in paths[:n_subjects]:
        with h5py.File(p, "r") as h:
            img = np.asarray(h["img"])                       # (n,H,W,C)
            mk = np.asarray(h["mask"])[..., 0]               # (n,H,W)
            has_et = "et" in h
            n_missing += (not has_et)
            et = np.asarray(h["et"]) if has_et else np.zeros_like(mk)
            if et.ndim == 4:                                  # tolerate (n,H,W,1) vs (n,H,W)
                et = et[..., 0]
            for i in range(0, img.shape[0], step):
                if mk[i].mean() < min_brain:
                    continue
                X0.append(img[i, ..., X0_IDX]); X1.append(img[i, ..., X1_IDX])
                MK.append(mk[i]); ET.append(et[i])
    if n_missing:
        print(f"[warn] {n_missing}/{min(n_subjects, len(paths))} subjects have no 'et' dataset; "
              f"their slices contribute 0 ET voxels")
    f32 = lambda L: np.asarray(L, dtype=np.float32)
    return f32(X0), f32(X1), f32(MK), f32(ET)


x0, x1, mask, et = sample_slices(VAL_ROOT)
print(f"sampled {x0.shape[0]} slices of {x0.shape[1]}x{x0.shape[2]} from {N_SUBJECTS} subjects")

m = mask > 0.5
me = (et > 0.5) & m
r = x0 - x1
rms = lambda a: float(np.sqrt(np.mean(a ** 2)))

VARSIGMA    = rms(r[m])                                    # the number that sets the schedule
VARSIGMA_ET = rms(r[me]) if me.sum() else float("nan")
DATA_RANGE  = float(np.percentile(x0[m], 99.5) - np.percentile(x0[m], 0.5))

print(f"\n  brain voxels {m.sum():,}   ET voxels {me.sum():,} "
      f"({me.sum()/max(m.sum(),1):.3%} of brain)")
print(f"  RMS(x0)        = {rms(x0[m]):.4f}      (T1ce, brain)")
print(f"  RMS(x1)        = {rms(x1[m]):.4f}      (T1,   brain)")
print(f"  varsigma       = {VARSIGMA:.4f}      <-- RMS(x0-x1) over the BRAIN")
print(f"  varsigma (ET)  = {VARSIGMA_ET:.4f}      <-- RMS(x0-x1) inside the ENHANCING TUMOR")
print(f"  data_range     = {DATA_RANGE:.4f}      (p99.5-p0.5 of x0, used for PSNR below)")

per_slice = np.array([rms(r[i][m[i]]) for i in range(len(r)) if m[i].sum() > 0])
sub = np.random.choice(r[m], size=min(400_000, int(m.sum())), replace=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(per_slice, bins=40, color="steelblue")
ax[0].axvline(VARSIGMA, color="k", ls="--", label=f"pooled {VARSIGMA:.3f}")
ax[0].set_xlabel(r"per-slice RMS$(x_0-x_1)$"); ax[0].set_ylabel("slices"); ax[0].legend()
ax[0].set_title("a single scalar hides this much spread")
ax[1].hist(sub, bins=200, color="indianred", log=True)
ax[1].set_xlabel(r"$x_0-x_1$ (brain voxels)"); ax[1].set_ylabel("count (log)")
ax[1].set_title("residual is sparse + heavy-tailed, not Gaussian")
plt.tight_layout(); plt.show()

## Part 4 - the criterion

With $x_1$ as prior mean and residual power $\varsigma^2$, the best **linear** estimate from
$u = x_0 + \sigma\varepsilon$ is

$$\hat x_0 = \alpha u + (1-\alpha)x_1, \qquad
\alpha^\star(\sigma) = \frac{\varsigma^2}{\varsigma^2+\sigma^2}, \qquad
\mathrm{MSE}^\star = \frac{\varsigma^2\sigma^2}{\varsigma^2+\sigma^2}.$$

$\alpha^\star = \tfrac12$ exactly when $\sigma_{\mathrm{eff}} = \varsigma$: the bridge state and the
prior are equally informative. **Rule: pick `beta_max` so that crossover lands at $t^\star \approx
0.5$**, splitting uniformly-sampled training steps evenly between the prior-dominated and
data-dominated regimes. Since the crossover reaches $t^\star = 0.5$ when $\sqrt{V} = \varsigma$,
this is just **`beta_max` such that `std_fwd[-1] == varsigma`**.

In [ ]:
def solve_beta_max(target, n_points=N_POINTS, lo=1e-4, hi=1e5, iters=200):
    """beta_max whose schedule ends at std_fwd[-1] == target. Monotone in beta_max -> bisect in
    log space. Returns (beta_max, hit_floor): the fixed linear_start=1e-4 puts a FLOOR on the
    total variance, so targets below it are unreachable with this schedule family."""
    f = lambda b: float(build_schedule(kind="i2sb", n_points=n_points, beta_max=b).std_fwd[-1])
    if not np.isfinite(target):
        return float("nan"), False
    if f(lo) >= target:
        return lo, True
    for _ in range(iters):
        mid = math.sqrt(lo * hi)
        if f(mid) < target: lo = mid
        else:               hi = mid
    return math.sqrt(lo * hi), False


BETA_BRAIN, floor_b = solve_beta_max(VARSIGMA)
BETA_ET, floor_e = solve_beta_max(VARSIGMA_ET)
print(f"centred on whole-brain varsigma={VARSIGMA:.4f}  ->  beta_max = {BETA_BRAIN:.3f}"
      + ("   [!] AT THE linear_start FLOOR -- the schedule cannot go this quiet" if floor_b else ""))
print(f"centred on ET         varsigma={VARSIGMA_ET:.4f}  ->  beta_max = {BETA_ET:.3f}"
      + ("   [!] at the floor" if floor_e else ""))
print(f"the paper default beta_max={BETA_REF} centres on varsigma={math.sqrt(V_ref):.4f}")

CANDS = sorted(set(BETA_GRID + [round(BETA_BRAIN, 3), round(BETA_ET, 3)]))
print(f"\n{'beta_max':>9} {'t* (brain)':>11} {'t* (ET)':>9} {'peak std_sb':>12}")
for b in CANDS:
    s = build_schedule(kind="i2sb", n_points=N_POINTS, beta_max=b)
    _, _, std_sb = bridge_coeffs(s)
    print(f"{b:>9.3f} {crossover_t(s, VARSIGMA):>11.3f} {crossover_t(s, VARSIGMA_ET):>9.3f} "
          f"{float(std_sb.max()):>12.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for b in CANDS:
    s = build_schedule(kind="i2sb", n_points=N_POINTS, beta_max=b)
    se = sigma_eff(s)
    ax[0].semilogy(t, se, label=f"{b:g}")
    ax[1].plot(t, VARSIGMA ** 2 / (VARSIGMA ** 2 + se ** 2), label=f"{b:g}")
ax[0].axhline(VARSIGMA, color="k", ls="--", lw=1.5, label=r"$\varsigma$ brain")
if np.isfinite(VARSIGMA_ET):
    ax[0].axhline(VARSIGMA_ET, color="r", ls=":", lw=1.5, label=r"$\varsigma$ ET")
ax[0].set_ylabel(r"$\sigma_{eff}$"); ax[0].set_title("ladder vs data scale")
ax[1].axhline(0.5, color="k", ls="--", lw=1)
ax[1].set_ylabel(r"$\alpha^\star$ (weight on the bridge state)")
ax[1].set_title(r"$\alpha^\star>\frac{1}{2}$: $x_t$ beats the prior")
for a in ax:
    a.set_xlabel("t"); a.grid(alpha=.3); a.legend(fontsize=7, ncol=2, title="beta_max")
plt.tight_layout(); plt.show()

## Part 5 - empirical check on real slices

The algebra above is Gaussian; BraTS residuals are not (see the log-histogram in Part 3 -- sparse,
heavy-tailed at the enhancement). So measure, on real voxels, what two **training-free** estimators
recover from $u(t)$ at each rung:

1. **prior only** &mdash; $\hat x_0 = x_1$. Flat in $t$; the floor any bridge must beat.
2. **linear Bayes** &mdash; $\alpha^\star u + (1-\alpha^\star)x_1$ with the measured $\varsigma$.
3. **soft-threshold** &mdash; $x_1 + \mathrm{ST}(u-x_1,\lambda)$, $\lambda$ oracle-tuned per rung.
   Exploits the residual's *sparsity*, which is exactly the structure a CDLNet/GroupCDL regressor is
   built to use, so the gap between 2 and 3 is the non-Gaussian headroom a learned net can claim.

Where curve 3 pulls away from curve 1 is where the schedule is doing useful work.

In [ ]:
T_GRID = np.linspace(0.02, 0.98, 25)


def psnr(mse, data_range=None):
    dr = DATA_RANGE if data_range is None else data_range
    return 10.0 * np.log10(dr ** 2 / max(float(mse), 1e-12))


def _region_residual(region, seed=0):
    """Brain/ET residual voxels as a 1-D tensor, subsampled to MAX_VOX for speed."""
    res = torch.from_numpy((x0 - x1)[region])
    if res.numel() > MAX_VOX:
        g = torch.Generator().manual_seed(seed)
        res = res[torch.randperm(res.numel(), generator=g)[:MAX_VOX]]
    return res


def evaluate(beta_max, region, varsigma, n_lam=21):
    """Measured MSE of the three estimators across the t grid, on `region` voxels."""
    se = sigma_eff(build_schedule(kind="i2sb", n_points=N_POINTS, beta_max=beta_max))
    resid = _region_residual(region)
    g = torch.Generator().manual_seed(0)
    out = {k: [] for k in ("sigma_eff", "prior", "linear", "soft")}
    for tt in T_GRID:
        sig = float(se[min(int(tt * N_POINTS), N_POINTS - 1)])
        # u - x1 = (x0 - x1) + sigma_eff * eps : the exact sufficient statistic (see header)
        u_res = resid + sig * torch.randn(resid.shape, generator=g)
        a = varsigma ** 2 / (varsigma ** 2 + sig ** 2)
        soft = min(float((torch.sign(u_res) * (u_res.abs() - l).clamp(min=0) - resid).pow(2).mean())
                   for l in np.linspace(0.0, 3.0 * sig, n_lam))
        out["sigma_eff"].append(sig)
        out["prior"].append(float(resid.pow(2).mean()))              # x_hat = x1
        out["linear"].append(float((a * u_res - resid).pow(2).mean()))
        out["soft"].append(soft)
    return {k: np.asarray(v) for k, v in out.items()}


SHOW = sorted(set([round(BETA_BRAIN, 3), BETA_REF, 1.0] +
                  ([round(BETA_ET, 3)] if np.isfinite(BETA_ET) else [])))
REGIONS = [("brain", m, VARSIGMA)] + ([("ET", me, VARSIGMA_ET)] if me.sum() else [])

fig, axes = plt.subplots(len(REGIONS), len(SHOW), figsize=(3.5 * len(SHOW), 3.5 * len(REGIONS)),
                         sharey="row", squeeze=False)
for j, b in enumerate(SHOW):
    s = build_schedule(kind="i2sb", n_points=N_POINTS, beta_max=b)
    for i, (name, reg, vs) in enumerate(REGIONS):
        e = evaluate(b, reg, vs)
        a = axes[i][j]
        a.plot(T_GRID, [psnr(v) for v in e["prior"]], "k--", label="prior only ($x_1$)")
        a.plot(T_GRID, [psnr(v) for v in e["linear"]], "-o", ms=3, label="linear Bayes")
        a.plot(T_GRID, [psnr(v) for v in e["soft"]], "-s", ms=3, label="soft-threshold")
        a.axvline(crossover_t(s, vs), color="r", ls=":", lw=1.5, label=r"$\sigma_{eff}=\varsigma$")
        a.set_title(f"beta_max={b:g}  [{name}]", fontsize=10)
        a.set_xlabel("t"); a.grid(alpha=.3)
        if j == 0: a.set_ylabel(f"{name} PSNR (dB)")
        if i == 0 and j == 0: a.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Part 6 - verdict

Careful with the obvious scalar here. "Mean dB recovered over the prior" is **degenerate**: less
noise always recovers more, so it is maximized by $\sigma_{\mathrm{eff}}\to 0$ -- the deterministic
interpolation limit, i.e. the one regime we are trying to avoid. It would always tell you to pick
the smallest `beta_max`.

The non-degenerate question is how the schedule *distributes* its steps. Normalize each rung by what
the prior alone achieves:

$$q(t) \;=\; \frac{\mathrm{MSE}_{\text{prior}} - \mathrm{MSE}_{\text{soft}}(t)}{\mathrm{MSE}_{\text{prior}}} \in [0,1]$$

$q \to 1$ is a **trivial** step (the state already hands over $x_0$); $q \to 0$ is a **useless** step
(no better than answering $x_1$). Every schedule sweeps $q$ from 1 down to 0 -- what `beta_max`
controls is *where in $t$* that transition happens and how much of the schedule it occupies. So
score by the fraction of uniformly-sampled steps that are neither trivial nor useless. That has a
genuine interior optimum.

In [ ]:
TRIVIAL, USELESS = 0.9, 0.1     # q > 0.9 = step learns nothing new; q < 0.1 = step is hopeless


def step_budget(beta_max, region, varsigma):
    """(% trivial, % informative, % useless) over uniformly sampled t."""
    e = evaluate(beta_max, region, varsigma)
    q = 1.0 - e["soft"] / np.maximum(e["prior"], 1e-12)
    return (100 * float(np.mean(q > TRIVIAL)),
            100 * float(np.mean((q >= USELESS) & (q <= TRIVIAL))),
            100 * float(np.mean(q < USELESS)))


rows = []
for b in CANDS:
    s = build_schedule(kind="i2sb", n_points=N_POINTS, beta_max=b)
    br = step_budget(b, m, VARSIGMA)
    etb = step_budget(b, me, VARSIGMA_ET) if me.sum() else (float("nan"),) * 3
    rows.append((b, crossover_t(s, VARSIGMA), br, etb))

print("                          ---------- brain ----------   ----------- ET -----------")
print(f"{'beta_max':>9} {'t* brain':>9} {'trivial':>9} {'INFORM':>8} {'useless':>8}"
      f" {'trivial':>10} {'INFORM':>8} {'useless':>8}")
for b, ts, br, etb in rows:
    print(f"{b:>9.3f} {ts:>9.3f} {br[0]:>8.0f}% {br[1]:>7.0f}% {br[2]:>7.0f}%"
          f" {etb[0]:>9.0f}% {etb[1]:>7.0f}% {etb[2]:>7.0f}%")

best_brain = max(rows, key=lambda r: r[2][1])[0]
best_et = max(rows, key=lambda r: (r[3][1] if np.isfinite(r[3][1]) else -1))[0]
print(f"\nmost informative steps, brain : beta_max = {best_brain:g}")
print(f"most informative steps, ET    : beta_max = {best_et:g}")
print(f"criterion (t*=0.5, whole brain): beta_max = {BETA_BRAIN:.2f}")
print(f"criterion (t*=0.5, ET)         : beta_max = {BETA_ET:.2f}")
print(f"paper default                  : beta_max = {BETA_REF} "
      f"(centres on varsigma={math.sqrt(V_ref):.3f})")

# A maximum sitting on the edge of CANDS is not a maximum -- the statistic is still climbing when
# the grid runs out (and beta_max ~ 0.1 is the schedule's own degenerate floor, see Part 1). Say so
# rather than quietly treating a boundary value as an optimum.
for lbl, best in (("brain", best_brain), ("ET", best_et)):
    if np.isfinite(best) and best in (min(CANDS), max(CANDS)):
        print(f"[!] the {lbl} optimum is at the EDGE of the candidate grid ({best:g}) -- it is a "
              f"boundary, not an interior optimum. Widen BETA_GRID before reading anything into it.")

fig, ax = plt.subplots(figsize=(6.5, 4))
bs = [r[0] for r in rows]
ax.semilogx(bs, [r[2][1] for r in rows], "-o", label="brain")
if me.sum():
    ax.semilogx(bs, [r[3][1] for r in rows], "-s", label="ET")
ax.axvline(BETA_BRAIN, color="k", ls="--", lw=1, label=r"$t^\star=0.5$ brain")
ax.axvline(BETA_REF, color="gray", ls=":", lw=1, label="paper default")
ax.set_xlabel("beta_max"); ax.set_ylabel("% informative steps")
ax.set_title("interior optimum: too small = trivial steps, too large = useless steps")
ax.grid(alpha=.3, which="both"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

# The DEFAULT is the t*=0.5 whole-brain criterion: it is the one number here that is principled,
# non-degenerate and derived purely from measured data. The informative-steps table above is a
# CHECK on it, not a competing estimate -- and where the two disagree, that disagreement is the
# reason to sweep rather than to average them into a number neither criterion actually supports.
RECOMMENDED = round(BETA_BRAIN, 2)
SWEEP_LO = min(BETA_BRAIN, best_brain)
SWEEP_HI = max(BETA_BRAIN, best_et if np.isfinite(best_et) else BETA_BRAIN)
sweep = [float(f"{v:.3g}") for v in np.geomspace(SWEEP_LO, max(SWEEP_HI, SWEEP_LO * 1.01), 5)]

print(f"\nRECOMMENDED beta_max = {RECOMMENDED}   (t*=0.5 on whole-brain varsigma)")
print(f"sweep range          = [{SWEEP_LO:.3g}, {SWEEP_HI:.3g}]  ->  {sweep}")
print(f"   lower end = whole brain (bulk anatomy), upper end = enhancing tumor.")
print(f"\n--- paste into config/BraTS/i2sb_gcdl_t1.json ---")
print(json.dumps({"i2sb": {"kind": "i2sb", "n_points": N_POINTS, "beta_max": RECOMMENDED}},
                 indent=4))
print("\n--- and for torch/i2sb_tau_sweep.sbatch, retargeted to beta_max ---")
print(f"TAUS=({' '.join(str(v) for v in sweep)})    # rename to BETAS + override i2sb.beta_max")

## What this settles, and what it does not

**Settles:** the *scale* of the schedule. `beta_max` sets the denoising ladder
$\sigma_{\mathrm{eff}}(t)$, and this notebook matches that ladder to the measured T1&rarr;T1ce gap so
uniformly sampled training steps land where there is signal to recover. That is an
estimation-theoretic question with a data-driven answer, and it needs no training.

**Does not settle:** the perception/distortion tradeoff. Whether the extra stochasticity buys
*realistic enhancement texture* rather than merely lower MSE depends on what the trained regressor
does with it, and cannot be read off the data. Two caveats to carry forward:

* the soft-threshold curve is a *lower* bound on a learned net (it uses only sparsity, no anatomy),
  so the genuinely useful band is likely wider than plotted;
* whole-brain and ET disagree, often by a lot. The ET residual is several times larger, so the ET
  side always wants a noisier schedule. Since the enhancing tumor is the entire point of T1ce, the
  disagreement is exactly why Part 6 emits a *range* rather than a single number -- its lower end is
  set by bulk anatomy, its upper end by the tumor.

To confirm, run a training probe over that range. `torch/i2sb_tau_sweep.sbatch` already implements
this pattern (copy config, override one key, own save_dir + wandb run): point `BASE_CONFIG` at
`config/BraTS/i2sb_gcdl_t1.json`, change the overridden key from `i2sb.tau` to `i2sb.beta_max`, set
`TAUS` to the beta_max grid, and set `SWEEP_EPOCHS` small to probe cheaply first.